# Hourly Rental Aggregation and Time Feratures

The goal of this notebook is to transform raw bike rental event records into a complete hourly demand dataset. 

The final output of this notebook includes two CSV files:

1. A full version with the holiday name column for checking and validation.
2. A model-ready version that only keeps the numeric `is_holiday` feature.

The notebook is structured as follow:

1. Prepare rental datasets
2. Merge registered and direct rental data
3. Merge with weather data
4. Merge with holiday data
5. Save final dataset

## 1. Prepare rental datasets
### 1.1 Load and convert datetime

In [326]:
# this has been done in the first part
import pandas as pd

registered = pd.read_csv("../data/registered_bike_rentals.csv")
direct = pd.read_csv("../data/direct_pickup_bike_rentals.csv")
registered["datetime"] = pd.to_datetime(registered["datetime"])
direct["datetime"] = pd.to_datetime(direct["datetime"])

### 1.2 Round datetime to hour

In [327]:
# each timestamps is floored to the beginning of its hour
registered["hour"] = registered["datetime"].dt.floor("h")
direct["hour"] = direct["datetime"].dt.floor("h")

### 1.3 Group rental events by hour

In [328]:
registered_hourly = (
    registered
    .groupby("hour")
    .size()
    .reset_index(name="registered_count")
)

registered_hourly.head()

,hour,registered_count
0,2011-01-01 00:00:00,13
1,2011-01-01 01:00:00,32
2,2011-01-01 02:00:00,27
3,2011-01-01 03:00:00,10
4,2011-01-01 04:00:00,1


In [329]:
direct_hourly = (
    direct
    .groupby("hour")
    .size()
    .reset_index(name="direct_count")
)

direct_hourly.head()

,hour,direct_count
0,2011-01-01 00:00:00,3
1,2011-01-01 01:00:00,8
2,2011-01-01 02:00:00,5
3,2011-01-01 03:00:00,3
4,2011-01-01 06:00:00,2


## 2. Merge registered and direct hourly data
### 2.1 Combine hourly counts

In [330]:
# outer merge to show missing values
hourly_rentals = registered_hourly.merge(
    direct_hourly,
    on="hour",
    how="outer"
)

hourly_rentals.head()

,hour,registered_count,direct_count
0,2011-01-01 00:00:00,13.0,3.0
1,2011-01-01 01:00:00,32.0,8.0
2,2011-01-01 02:00:00,27.0,5.0
3,2011-01-01 03:00:00,10.0,3.0
4,2011-01-01 04:00:00,1.0,NaN


It shows there are missing counts at certain time in either of the two rental types. So we replace them with 0 to avoid unkown data:

In [331]:
# fill NaN with 0
hourly_rentals[["registered_count", "direct_count"]] = (
    hourly_rentals[["registered_count", "direct_count"]].fillna(0).astype(int)
)

# create the total rental count
hourly_rentals["total_count"] = (
    hourly_rentals["registered_count"] + hourly_rentals["direct_count"]
)

hourly_rentals = hourly_rentals.sort_values("hour").reset_index(drop=True) # sort by hours
hourly_rentals.head()

,hour,registered_count,direct_count,total_count
0,2011-01-01 00:00:00,13,3,16
1,2011-01-01 01:00:00,32,8,40
2,2011-01-01 02:00:00,27,5,32
3,2011-01-01 03:00:00,10,3,13
4,2011-01-01 04:00:00,1,0,1


### 2.2 Handle missing hourly records
From the exploration part we have already known that, there are missing hourly records in the rental data. Therefore we create a full time table, find the specific rows and fill the values with 0.

In [332]:
print("Index number before handling:", hourly_rentals.shape[0])

Index number before handling: 17379


In [333]:
# create full hours table
full_hours = pd.DataFrame({
    "hour": pd.date_range(
        start=hourly_rentals["hour"].min(),
        end=hourly_rentals["hour"].max(),
        freq="h"
    )
})

#merge
hourly_rentals = full_hours.merge(
    hourly_rentals,
    on="hour",
    how="left"
)

# mark missing rows
hourly_rentals["was_missing_rental"] = hourly_rentals["total_count"].isna().astype(int)

# fill up
count_cols = ["registered_count", "direct_count", "total_count"]
hourly_rentals[count_cols] = (
    hourly_rentals[count_cols]
    .fillna(0)
    .astype(int)
)

# show missing hours
hourly_rentals[hourly_rentals["was_missing_rental"] == 1]

,hour,registered_count,direct_count,total_count,was_missing_rental
29,2011-01-02 05:00:00,0,0,0,1
50,2011-01-03 02:00:00,0,0,0,1
51,2011-01-03 03:00:00,0,0,0,1
75,2011-01-04 03:00:00,0,0,0,1
99,2011-01-05 03:00:00,0,0,0,1
...,...,...,...,...,...
16044,2012-10-30 12:00:00,0,0,0,1
16251,2012-11-08 03:00:00,0,0,0,1
16755,2012-11-29 03:00:00,0,0,0,1
17356,2012-12-24 04:00:00,0,0,0,1


In [334]:
print("Expected rows from full_hours:", full_hours.shape[0])
print("Index number after handling:", hourly_rentals.shape[0])

Expected rows from full_hours: 17544
Index number after handling: 17544


### 2.3 Derive time-based features

In [335]:
hourly_rentals["date"] = hourly_rentals["hour"].dt.date  # for holidays merging
hourly_rentals["hour_of_day"] = hourly_rentals["hour"].dt.hour # demand related: rush hour, lunch time..
hourly_rentals["day_of_week"] = hourly_rentals["hour"].dt.dayofweek # weekdays and weekends
hourly_rentals["month"] = hourly_rentals["hour"].dt.month # seasonal demand
hourly_rentals["is_weekend"] = hourly_rentals["day_of_week"].isin([5, 6]).astype(int)

hourly_rentals.head()

,hour,registered_count,direct_count,total_count,was_missing_rental,date,hour_of_day,day_of_week,month,is_weekend
0,2011-01-01 00:00:00,13,3,16,0,2011-01-01,0,5,1,1
1,2011-01-01 01:00:00,32,8,40,0,2011-01-01,1,5,1,1
2,2011-01-01 02:00:00,27,5,32,0,2011-01-01,2,5,1,1
3,2011-01-01 03:00:00,10,3,13,0,2011-01-01,3,5,1,1
4,2011-01-01 04:00:00,1,0,1,0,2011-01-01,4,5,1,1


In [336]:
hourly_rentals.info()

<class 'pandas.DataFrame'>
RangeIndex: 17544 entries, 0 to 17543
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   hour                17544 non-null  datetime64[us]
 1   registered_count    17544 non-null  int64         
 2   direct_count        17544 non-null  int64         
 3   total_count         17544 non-null  int64         
 4   was_missing_rental  17544 non-null  int64         
 5   date                17544 non-null  object        
 6   hour_of_day         17544 non-null  int32         
 7   day_of_week         17544 non-null  int32         
 8   month               17544 non-null  int32         
 9   is_weekend          17544 non-null  int64         
dtypes: datetime64[us](1), int32(3), int64(5), object(1)
memory usage: 1.1+ MB


In [337]:
hourly_rentals.isna().sum() # no missing value

hour                  0
registered_count      0
direct_count          0
total_count           0
was_missing_rental    0
date                  0
hour_of_day           0
day_of_week           0
month                 0
is_weekend            0
dtype: int64

In [338]:
print("Original registered records:", len(registered))
print("Aggregated registered count:", hourly_rentals["registered_count"].sum())

print("\nOriginal direct records:", len(direct))
print("Aggregated direct count:", hourly_rentals["direct_count"].sum())

print("\nOriginal total records:", len(registered) + len(direct))
print("Aggregated total count:", hourly_rentals["total_count"].sum())

Original registered records: 2672662
Aggregated registered count: 2672662

Original direct records: 620017
Aggregated direct count: 620017

Original total records: 3292679
Aggregated total count: 3292679


## 3. Merge with weather data
### 3.1 Select weather features and create a clean table

In [339]:
weather = pd.read_csv("../data/weather.csv")
weather["datetime"] = pd.to_datetime(weather["datetime"])

# create same column "hour" for merging
weather["hour"] = weather["datetime"].dt.floor("h")

# remove unnecessary features
weather_features = weather.drop(columns=["id", "datetime"])

weather_features.head()

,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh,hour
0,clear,3.3,3.0,81.0,0.0,2011-01-01 00:00:00
1,clear,2.3,2.0,80.0,0.0,2011-01-01 01:00:00
2,clear,2.3,2.0,80.0,0.0,2011-01-01 02:00:00
3,clear,3.3,3.0,75.0,0.0,2011-01-01 03:00:00
4,clear,3.3,3.0,75.0,0.0,2011-01-01 04:00:00


### 3.2 Merge with hourly_rentals

In [340]:
# left merge: hourly_rentals LEFT JOIN weather_features ON hour
rentals_with_weather = hourly_rentals.merge(
    weather_features,
    on="hour",
    how="left"
)
rentals_with_weather = rentals_with_weather.sort_values("hour").reset_index(drop=True)
# rentals_with_weather.head()

In [341]:
rentals_with_weather.info()
rentals_with_weather.shape

<class 'pandas.DataFrame'>
RangeIndex: 17544 entries, 0 to 17543
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   hour                     17544 non-null  datetime64[us]
 1   registered_count         17544 non-null  int64         
 2   direct_count             17544 non-null  int64         
 3   total_count              17544 non-null  int64         
 4   was_missing_rental       17544 non-null  int64         
 5   date                     17544 non-null  object        
 6   hour_of_day              17544 non-null  int32         
 7   day_of_week              17544 non-null  int32         
 8   month                    17544 non-null  int32         
 9   is_weekend               17544 non-null  int64         
 10  conditions               17379 non-null  str           
 11  temperature_c            17379 non-null  float64       
 12  perceived_temperature_c  17379 non-null  fl

(17544, 15)

### 3.3 handle missing weather values

In [342]:
weather_columns = [
    col for col in rentals_with_weather.columns
    if col not in [
        "hour",
        "registered_count",
        "direct_count",
        "total_count",
        "was_missing_rental",
    ]
]

# mark NaN as 1
rentals_with_weather["was_missing_weather"] = (
    rentals_with_weather[weather_columns].isna().any(axis=1).astype(int)
)

print("Rows with missing weather:", rentals_with_weather["was_missing_weather"].sum())
print(rentals_with_weather[weather_columns].isna().sum())

Rows with missing weather: 165
date                         0
hour_of_day                  0
day_of_week                  0
month                        0
is_weekend                   0
conditions                 165
temperature_c              165
perceived_temperature_c    165
humidity                   165
windspeed_kmh              165
dtype: int64


In [343]:
numeric_weather_cols = (
    rentals_with_weather[weather_columns]
    .select_dtypes(include="number")
    .columns
    .tolist()
)

categorical_weather_cols = (
    rentals_with_weather[weather_columns]
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

# print("Numeric weather columns:", numeric_weather_cols)
# print("Categorical weather columns:", categorical_weather_cols)

# fill up numerical columns with linear intepolate 
rentals_with_weather[numeric_weather_cols] = (
    rentals_with_weather[numeric_weather_cols]
    .interpolate(method="linear")
    .ffill()
    .bfill()
)

# fill up categorical columns 
rentals_with_weather[categorical_weather_cols] = (
    rentals_with_weather[categorical_weather_cols]
    .ffill()
    .bfill()
)
print(rentals_with_weather[weather_columns].isna().sum())

date                       0
hour_of_day                0
day_of_week                0
month                      0
is_weekend                 0
conditions                 0
temperature_c              0
perceived_temperature_c    0
humidity                   0
windspeed_kmh              0
dtype: int64


Missing weather values are handled differently from missing rental counts. Numeric weather columns such as `temperature_c`, `perceived_temperature_c`, `humidity`, `windspeed_kmh` are filled using time-based interpolation, while categorical weather columns like `conditions` are filled using forward and backward filling.

## 4. Merge with holiday data

In [344]:
holidays = pd.read_csv("../data/holidays.csv")
holidays["date"] = pd.to_datetime(holidays["date"]).dt.date #convert to same datatype "object" as main data

final_data = rentals_with_weather.merge(
    holidays[["date", "holiday"]],
    on="date",
    how="left"
)

final_data["is_holiday"] = final_data["holiday"].notna().astype(int)

## 5. Check final data

In [345]:
final_data.head()

,hour,registered_count,direct_count,total_count,was_missing_rental,date,hour_of_day,day_of_week,month,is_weekend,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh,was_missing_weather,holiday,is_holiday
0,2011-01-01 00:00:00,13,3,16,0,2011-01-01,0,5,1,1,clear,3.3,3.0,81.0,0.0,0,NaN,0
1,2011-01-01 01:00:00,32,8,40,0,2011-01-01,1,5,1,1,clear,2.3,2.0,80.0,0.0,0,NaN,0
2,2011-01-01 02:00:00,27,5,32,0,2011-01-01,2,5,1,1,clear,2.3,2.0,80.0,0.0,0,NaN,0
3,2011-01-01 03:00:00,10,3,13,0,2011-01-01,3,5,1,1,clear,3.3,3.0,75.0,0.0,0,NaN,0
4,2011-01-01 04:00:00,1,0,1,0,2011-01-01,4,5,1,1,clear,3.3,3.0,75.0,0.0,0,NaN,0


In [346]:
final_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 17544 entries, 0 to 17543
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   hour                     17544 non-null  datetime64[us]
 1   registered_count         17544 non-null  int64         
 2   direct_count             17544 non-null  int64         
 3   total_count              17544 non-null  int64         
 4   was_missing_rental       17544 non-null  int64         
 5   date                     17544 non-null  object        
 6   hour_of_day              17544 non-null  int32         
 7   day_of_week              17544 non-null  int32         
 8   month                    17544 non-null  int32         
 9   is_weekend               17544 non-null  int64         
 10  conditions               17544 non-null  str           
 11  temperature_c            17544 non-null  float64       
 12  perceived_temperature_c  17544 non-null  fl

In [347]:
# confirm all holidays matched
matched_holiday_dates = final_data[final_data["is_holiday"] == 1]["date"].nunique()
original_holiday_dates = holidays["date"].nunique()

print("Matched holiday dates:", matched_holiday_dates)
print("Original holiday dates:", original_holiday_dates)

Matched holiday dates: 21
Original holiday dates: 21


In [348]:
# final_data[final_data["is_holiday"] == 1].head() # shows only holidays
final_data[final_data["is_holiday"] == 1].groupby(["date", "holiday"]).size()

date        holiday                               
2011-01-17  Dr. Martin Luther King, Jr.'s Birthday    24
2011-02-21  Washington's Birthday                     24
2011-04-15  D.C. Emancipation Day (observed)          24
2011-05-30  Memorial Day                              24
2011-07-04  Independence Day                          24
2011-09-05  Labor Day                                 24
2011-10-10  Columbus Day                              24
2011-11-11  Veterans Day                              24
2011-11-24  Thanksgiving Day                          24
2011-12-26  Christmas Day (observed)                  24
2012-01-02  New Year's Day (observed)                 24
2012-01-16  Dr. Martin Luther King, Jr.'s Birthday    24
2012-02-20  Washington's Birthday                     24
2012-04-16  D.C. Emancipation Day                     24
2012-05-28  Memorial Day                              24
2012-07-04  Independence Day                          24
2012-09-03  Labor Day                

## 6. Save final dataset

In [349]:
# a full version with holiday names
final_data.to_csv("../data/final_data_with_holiday_name.csv", index=False)


# a model-ready version 
model_data = final_data.drop(columns=["holiday"])
model_data.to_csv("../data/final_model_data.csv", index=False)

## 7. Conclusion
The merging process successfully transformed the raw rental event records into a complete hourly modeling dataset. The final dataset contains the hourly rental demand, time-based features, weather features, and holiday information.


The registered and direct rental records were first aggregated by hour and combined into one hourly rental demand table. Missing rental hours were explicitly added using a complete hourly index and filled with `0`, because no rental records during an hour can be interpreted as zero rental demand.

The hourly rental data was then enriched with weather information. Missing weather values were not filled with `0`, because weather values such as temperature, humidity, and windspeed have real numerical meaning. Instead, numeric weather features were filled using interpolation, and categorical weather conditions were filled using forward and backward filling.

Finally, the holiday table was used as a lookup table to create the `is_holiday` feature. The holiday name column is useful for checking the merge result, but the final model should mainly use the numeric `is_holiday` feature.

Overall, the final dataset which is saved as a CSV file, is ready for the Dagster asset pipeline.